In [1]:
import pandas as pd

In [3]:
# df_posts = pd.read_excel("./2023_Completo_redem_0304.xlsx")
# df = df_posts.dropna(subset=['Message', 'Message-ID'])

df_res = pd.read_excel("./df_res_2.xlsx")
df_res = df_res.sort_values(by='min_distance')

df_factcheck = pd.read_excel("./Analise manual Aos fatos_v2.xlsx")
df_factcheck = df_factcheck.dropna(subset=['Resumo', 'resumo_2'])

In [11]:
from langchain_core.documents import (
    Document,
)  # Importa a classe Document (texto + metadados)

# Documentos de exemplo para os testes
docs_1 = [
    Document(
        page_content=row["Resumo"],
        metadata={
            # "source": "2023_Completo_redem_0304.xlsx"
            # "message_id": row["Message-ID"],
            "source": "factcheck_v1",
            "index": idx
        }
    )
    for idx, row in df_factcheck.iterrows()
]
docs = [
    Document(
        page_content=row["resumo_2"],
        metadata={
            # "source": "2023_Completo_redem_0304.xlsx"
            # "message_id": row["Message-ID"],
            "source": "factcheck_v2",
            "index": idx
        }
    )
    for idx, row in df_factcheck.iterrows()
]

In [12]:
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from embeders import SentenceTransformerEmbeddingFunction


# Retriever de palavra-chave (Esparso)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2


# Initialize ChromaDB (denso)
embeddings = SentenceTransformerEmbeddingFunction()
vectorstore = Chroma(
    persist_directory="./.chroma_db",
    collection_name="checks_v2_rewrite_cosine_sentence_transformer",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embeddings,
)
chroma_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 2, "score_threshold": 0.80},
)


# Ensemble Retriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever], weights=[0.5, 0.5]
)

In [6]:
idx = 11

message = df_res['message'].reset_index(drop=True)[idx]

print(f"MESSAGE: {message}\n\n")

res = ensemble_retriever.invoke(
    message,
    k=2,
)

print(f"TOTAL DE DOCUMENTOS: {len(res)}")
for r in res:
    print('\n')
    print("-"*100)
    print(r)

MESSAGE: Surpreendente: Governo Lula aprova recorde histórico de R$ 16,3 bilhões para projetos culturais via Lei Rouanet em 2023, contrastando com ajuste fiscal em andamento, inclusive com aumento de impostos, para evitar um rombo de R$ 168 bilhões no ano que vem. O vídeo revela o apoio fervoroso da classe artística ao governo, enquanto desafios econômicos persistem. 

Uma situação lamentável!




No relevant docs were retrieved using the relevance score threshold 0.8


TOTAL DE DOCUMENTOS: 2


----------------------------------------------------------------------------------------------------
page_content='Lei Rouanet (lei nº 8.313/1991) em 2023,  governo federal liberou  R$ 16 bilhões para artistas no fim do ano passado.' metadata={'source': 'factcheck_v2', 'index': 21}


----------------------------------------------------------------------------------------------------
page_content='mudança de sexo em crianças com verba da União' metadata={'source': 'factcheck_v2', 'index': 19}


In [10]:
# Helper function to get distances/scores from individual retrievers
# and create a wrapper that works with the ensemble retriever


def get_retrieval_with_distances(
        query: str,
        bm25_retriever,
        vectorstore,
        k: int = 10,
        max_vector_distance: float = None,
        bm25_weight: float = 0.5,
        vector_weight: float = 0.5,
        tau: float = 0.5,
    ):
    """
    Get retrieval results with distances from both BM25 and vector retrievers.

    Args:
        query: Search query
        bm25_retriever: BM25Retriever instance
        vectorstore: Chroma vectorstore instance
        k: Number of results to retrieve
        max_vector_distance: Maximum cosine distance threshold (None = no filter)
                            For cosine: 0=identical, 1=different
        bm25_weight: Weight for BM25 scores in combined ranking
        vector_weight: Weight for vector similarity in combined ranking
        tau: Threshold for combined score (from 0 to 1)

    Returns:
        List of dicts with: doc, bm25_score, vector_distance, vector_similarity, combined_score
    """
    # Get BM25 results with scores
    # LangChain's BM25Retriever may have the BM25 model accessible
    try:
        bm25_docs = bm25_retriever.invoke(query, k=k * 2)
        # Try to access the internal BM25 model
        from rank_bm25 import BM25Okapi

        tokenized_query = query.lower().split()

        # Check different possible attribute names for the BM25 model
        bm25_model = None
        if hasattr(bm25_retriever, "bm25"):
            bm25_model = bm25_retriever.bm25
        elif hasattr(bm25_retriever, "_model"):
            bm25_model = bm25_retriever._model
        elif hasattr(bm25_retriever, "vectorizer"):
            bm25_model = bm25_retriever.vectorizer

        if bm25_model and hasattr(bm25_model, "get_scores"):
            bm25_scores_all = bm25_model.get_scores(tokenized_query)
            # Map scores to documents
            all_docs_content = [doc.page_content for doc in docs]
            bm25_score_map = {}
            for idx, score in enumerate(bm25_scores_all):
                if idx < len(all_docs_content):
                    bm25_score_map[all_docs_content[idx]] = score
        else:
            # Fallback: reconstruct BM25 from documents if possible
            # Get all document texts
            all_docs_content = [doc.page_content for doc in docs]
            tokenized_texts = [text.lower().split() for text in all_docs_content]
            temp_bm25 = BM25Okapi(tokenized_texts)
            bm25_scores_all = temp_bm25.get_scores(tokenized_query)
            bm25_score_map = {
                all_docs_content[idx]: score
                for idx, score in enumerate(bm25_scores_all)
            }
    except Exception as e:
        print(f"Warning: Could not get BM25 scores: {e}")
        bm25_docs = []
        bm25_score_map = {}

    # Get vector search results with distances
    vector_results = vectorstore.similarity_search_with_score(query, k=k * 2)

    # Create a mapping of document content to scores
    results_dict = {}

    # Process BM25 results
    if bm25_score_map:
        for doc in bm25_docs:
            content = doc.page_content
            bm25_score = bm25_score_map.get(content, 0)

            if content not in results_dict:
                results_dict[content] = {
                    "doc": doc,
                    "bm25_score": bm25_score,
                    "vector_distance": 0,
                    "vector_similarity": 0,
                }
            else:
                results_dict[content]["bm25_score"] = bm25_score

    # Process vector results
    for doc, distance in vector_results:
        content = doc.page_content
        similarity = 1 - distance  # Convert distance to similarity

        if content not in results_dict:
            results_dict[content] = {
                "doc": doc,
                "bm25_score": 0,
                "vector_distance": distance,
                "vector_similarity": similarity,
            }
        else:
            results_dict[content]["vector_distance"] = distance
            results_dict[content]["vector_similarity"] = similarity

    # Normalize BM25 scores to [0, 1] range
    bm25_scores = [r["bm25_score"] for r in results_dict.values()]
    if bm25_scores and max(bm25_scores) > min(bm25_scores):
        max_bm25 = max(bm25_scores)
        min_bm25 = min(bm25_scores)
        bm25_range = max_bm25 - min_bm25
        for content in results_dict:
            results_dict[content]["bm25_score_normalized"] = (
                (results_dict[content]["bm25_score"] - min_bm25) / bm25_range
                if bm25_range > 0
                else 0
            )
    else:
        for content in results_dict:
            results_dict[content]["bm25_score_normalized"] = 0

    # Calculate combined score
    results = []
    for content, data in results_dict.items():
        # Filter by max_vector_distance if specified
        if max_vector_distance is not None and data["vector_distance"] is not None:
            if data["vector_distance"] > max_vector_distance:
                continue

        # Combined score: weighted sum of normalized BM25 and vector similarity
        combined_score = bm25_weight * data["bm25_score_normalized"] + vector_weight * (
            data["vector_similarity"] if data["vector_similarity"] is not None else 0
        )
        if combined_score < tau:
            continue

        results.append(
            {
                "doc": data["doc"],
                "bm25_score": data["bm25_score"],
                "bm25_score_normalized": data["bm25_score_normalized"],
                "vector_distance": data["vector_distance"],
                "vector_similarity": data["vector_similarity"],
                "combined_score": combined_score,
            }
        )

    # Sort by combined score (higher is better)
    results.sort(key=lambda x: x["combined_score"], reverse=True)

    return results[:k]


# Test the function
idx = 17

message = df_res['message'].reset_index(drop=True)[idx]
print(f"MESSAGE: {message}\n\n")

results = get_retrieval_with_distances(
    message,
    bm25_retriever,
    vectorstore,
    k=2,
    max_vector_distance=0.4,
    vector_weight=0.5,
    tau=0.3
)
results

MESSAGE: Na Câmara dos Deputados, fizemos história ao aprovar o destaque 7 da Lei de Diretrizes Orçamentárias (LDO).

Essa medida é um marco: agora, é proibido o emprego de fundos públicos para encorajar ou custear invasões de terras, a prática do aborto, qualquer intervenção cirúrgica voltada à mudança de gênero em crianças e adolescentes, ou atividades que desafiem os valores da família tradicional.

Colocando o respeito à vida e à propriedade como nossos maiores valores!




[]

In [23]:
from tqdm import tqdm

batch_size = 100
num_rows = len(df_res)

# Prepare placeholders for results
all_docs_hs_1 = []
all_docs_hs_2 = []

for start_idx in tqdm(range(0, num_rows, batch_size)):
    end_idx = min(start_idx + batch_size, num_rows)
    batch_messages = df_res['message'].iloc[start_idx:end_idx].tolist()

    # Retrieve results for the batch
    batch_res_1 = [ensemble_retriever.invoke(msg, k=2) for msg in batch_messages]
    batch_res_2 = [
        get_retrieval_with_distances(
            msg,
            bm25_retriever,
            vectorstore,
            k=2,
            max_vector_distance=0.4,
            vector_weight=0.5,
            tau=0.3
        )
        for msg in batch_messages
    ]
    all_docs_hs_1.extend(batch_res_1)
    all_docs_hs_2.extend(batch_res_2)

df_res['all_docs_hs_1'] = all_docs_hs_1
df_res['all_docs_hs_2'] = all_docs_hs_2

  0%|          | 0/917 [00:00<?, ?it/s]No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were retrieved using the relevance score threshold 0.8
No relevant docs were ret

In [25]:
df_res.to_excel("./df_res_hs.xlsx", index=False)

In [8]:
# Example: Get distances from individual retrievers
idx = 3

query = df_res['message'].reset_index(drop=True)[idx]


print(f"QUERY: {query}\n\n")
print("="*100)

# Option 1: Get vector distances directly from vectorstore
print("\n1. VECTOR SEARCH WITH DISTANCES:")
print("-"*100)
vector_results = vectorstore.similarity_search_with_score(query, k=5)
for i, (doc, distance) in enumerate(vector_results, 1):
    similarity = 1 - distance  # Convert distance to similarity
    print(f"\nResult {i}:")
    print(f"  Distance: {distance:.4f} (lower is better)")
    print(f"  Similarity: {similarity:.4f} (higher is better)")
    print(f"  Content: {doc.page_content[:100]}...")
    
# Option 2: Get BM25 results (access internal scores if possible)
print("\n\n2. BM25 SEARCH:")
print("-"*100)
bm25_docs = bm25_retriever.invoke(query, k=5)
for i, doc in enumerate(bm25_docs, 1):
    print(f"\nResult {i}:")
    print(f"  Content: {doc.page_content[:100]}...")

# Option 3: Use the helper function to get combined scores
print("\n\n3. ENSEMBLE WITH DISTANCES:")
print("-"*100)
results = get_retrieval_with_distances(
    query, 
    bm25_retriever, 
    vectorstore, 
    k=5,
    max_vector_distance=0.8  # Filter out results with distance > 0.8
)
for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"  Combined Score: {result['combined_score']:.4f}")
    print(f"  BM25 Score: {result['bm25_score']:.4f} (normalized: {result['bm25_score_normalized']:.4f})")
    if result['vector_distance'] is not None:
        print(f"  Vector Distance: {result['vector_distance']:.4f}")
        print(f"  Vector Similarity: {result['vector_similarity']:.4f}")
    print(f"  Content: {result['doc'].page_content[:100]}...")


QUERY: Estamos agora, no plenário do Senado para a Sessão de Debates Temáticos sobre o aborto, convocada pelo Senador eduardogiraooficial. 

Eu e meu marido, coronel_aginaldo não poderíamos encontrar com uma Senadora mais engajada na luta pela vida!

damaresalvesoficial1 fez história com seu incrível trabalho pelas mulheres, crianças, bebês, deficientes, idosos e vulneráveis no Ministério da Mulher, Família e Direitos Humanos de jairmessiasbolsonaro.

#AbortoNao



1. VECTOR SEARCH WITH DISTANCES:
----------------------------------------------------------------------------------------------------

Result 1:
  Distance: 0.3587 (lower is better)
  Similarity: 0.6413 (higher is better)
  Content: nota técnica do Ministério da Saúde ampliou o acesso ao aborto no Brasil...

Result 2:
  Distance: 0.3810 (lower is better)
  Similarity: 0.6190 (higher is better)
  Content: empresas utilizam fetos abortados para fabricar cosméticos, especialmente após a divulgação do PL 1....

Result 3:
  Dista

## Summary: Getting Distances from LangChain EnsembleRetriever

LangChain's `EnsembleRetriever` uses **Reciprocal Rank Fusion (RRF)** which doesn't directly expose similarity scores/distances. However, you have several options:

### 1. **Get distances from individual retrievers:**
- **Vector retriever**: Use `vectorstore.similarity_search_with_score(query, k=k)` to get `(Document, distance)` tuples
- **BM25 retriever**: Access internal BM25 model or reconstruct scores (see helper function above)

### 2. **Configure distance thresholds:**
- Filter vector results by `max_distance` (for cosine: 0=identical, 1=different)
- Use the `get_results_with_distance_threshold()` function for simple filtering
- Use `get_retrieval_with_distances()` for combined BM25 + vector with thresholds

### 3. **Configure retriever parameters:**
You can adjust the search parameters of individual retrievers:

```python
# Adjust vector retriever k parameter
chroma_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 20}  # Retrieve more candidates
)

# Adjust BM25 retriever
bm25_retriever.k = 20

# Create ensemble with adjusted retrievers
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.5, 0.5]
)
```

### 4. **Distance metrics:**
- **Cosine distance**: Range [0, 1] where 0=identical, 1=different
- **Cosine similarity**: Range [0, 1] where 1=identical, 0=different (similarity = 1 - distance)
- Your Chroma collection uses cosine distance (`"hnsw:space": "cosine"`)


In [ ]:
# Simple approach: Get distances directly from vectorstore and configure threshold

def get_results_with_distance_threshold(
    query: str,
    vectorstore,
    max_distance: float = 0.8,
    k: int = 10
):
    """
    Simple function to get vector search results filtered by distance threshold.
    
    Args:
        query: Search query
        vectorstore: Chroma vectorstore
        max_distance: Maximum cosine distance (0=identical, 1=different)
        k: Maximum number of results to return
    
    Returns:
        List of (Document, distance, similarity) tuples
    """
    results = vectorstore.similarity_search_with_score(query, k=k*2)
    filtered = [
        (doc, distance, 1 - distance) 
        for doc, distance in results 
        if distance <= max_distance
    ]
    return filtered[:k]

# Example usage
query_example = "São Paulo não está entre as cinco melhores cidades"
print(f"Query: {query_example}\n")
print("Results with distance threshold 0.7:")
print("-"*80)

filtered_results = get_results_with_distance_threshold(
    query_example,
    vectorstore,
    max_distance=0.7,  # Only return results with distance <= 0.7
    k=5
)

for i, (doc, distance, similarity) in enumerate(filtered_results, 1):
    print(f"\n{i}. Distance: {distance:.4f}, Similarity: {similarity:.4f}")
    print(f"   {doc.page_content[:150]}...")


In [ ]:
from rank_bm25 import BM25Okapi
from langchain_community.vectorstores import Chroma
from typing import List
import numpy as np


class BM25Retriever:
    def __init__(self, documents: List[Document]):
        texts = [doc.page_content for doc in documents]
        tokenized_texts = [text.lower().split() for text in texts]
        self.bm25 = BM25Okapi(tokenized_texts)
        self.documents = documents

    def get_relevant_documents(self, query: str, k: int = 2):
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[-k:][::-1]
        return [self.documents[i] for i in top_indices]


class EnsembleRetriever:
    def __init__(self, bm25_retriever, vectorstore, bm25_weight=0.5):
        self.bm25_retriever: BM25Retriever = bm25_retriever
        self.vectorstore: Chroma = vectorstore
        self.bm25_weight: float = bm25_weight
        self.vector_weight: float = 1 - bm25_weight

    def get_relevant_documents(self, query: str, k: int = 2):
        # Get BM25 results
        bm25_docs = self.bm25_retriever.get_relevant_documents(query, k=k)

        # Get vector search results
        vector_docs = self.vectorstore.similarity_search(query, k=k)

        # Combine scores
        doc_scores = {}
        for i, doc in enumerate(bm25_docs):
            content = doc.page_content
            if content not in doc_scores:
                doc_scores[content] = {"doc": doc, "score": 0}
            doc_scores[content]["score"] += (
                self.bm25_weight * (len(bm25_docs) - i) / len(bm25_docs)
            )

        for i, doc in enumerate(vector_docs):
            content = doc.page_content
            if content not in doc_scores:
                doc_scores[content] = {"doc": doc, "score": 0}
            doc_scores[content]["score"] += (
                self.vector_weight * (len(vector_docs) - i) / len(vector_docs)
            )

        # Sort by combined score and return top k
        sorted_docs = sorted(
            doc_scores.values(), key=lambda x: x["score"], reverse=True
        )
        return [item["doc"] for item in sorted_docs[:k]]


# Initialize BM25
bm25_retriever = BM25Retriever(docs)

# Initialize ChromaDB
embeddings = SentenceTransformerEmbeddingFunction()
vectorstore = Chroma(
    persist_directory="./.chroma_db", 
    collection_name="posts_cosine_sentence_transformer", 
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embeddings
)

# Create ensemble retriever
ensemble_retriever = EnsembleRetriever(bm25_retriever, vectorstore, bm25_weight=0.5)

In [10]:
vectorstore.similarity_search("lei rouanet", k=2)

[Document(metadata={}, page_content='Dia Mundial da Lei ??'),
 Document(metadata={}, page_content='E a Claudia Raia poderia doar os 5 milhões que conseguiu via Lei Rouanet ??')]

In [11]:
ensemble_retriever.get_relevant_documents("Lei rouanet")

[Document(metadata={'source': 'doc_ssh_acesso_remoto'}, page_content='SSH, ou Secure Shell, é um protocolo que permite acesso remoto seguro a servidores. Para conectar, utilize a porta padrão 22 e um cliente SSH.'),
 Document(metadata={}, page_content='Dia Mundial da Lei ??')]

# Teste

In [17]:
import pandas as pd

In [18]:
df_factcheck = pd.read_excel("./Analise manual Aos fatos_v2.xlsx")

In [22]:
idx = 8
resumo_1 = df_factcheck.loc[idx, 'Resumo']
resumo_2 = df_factcheck.loc[idx, 'resumo_2']


res = ensemble_retriever.get_relevant_documents(resumo_2, k=5)

print(f"MAIN TEXT: {resumo_2}\n\n")
print("="*100)
# for distance, doc_ in zip(*res["distances"], *res["documents"]):
#     print(f"Distance: {distance}, Document: {doc_}\n")
#     print("-"*100)
print(res)

MAIN TEXT: Houve mais residências sem luz em São Paulo do que na Flórida após o furacão Milton.


[Document(metadata={'source': 'doc_ssh_acesso_remoto'}, page_content='SSH, ou Secure Shell, é um protocolo que permite acesso remoto seguro a servidores. Para conectar, utilize a porta padrão 22 e um cliente SSH.'), Document(metadata={}, page_content='Hoje, as felicitações de mais um ano próspero vão para os meus amigos campo-larguenses e também para todos que residem e amam a nossa #CampoLargo. Contem comigo aqui em Brasilia, para lutar pelos nossos pequenos e médios municipios!\n\n#Municipalista #DeputadoGiacobo #Parana'), Document(metadata={}, page_content='Folha de São Paulo e UOL sempre justos com minha ações corajosas!'), Document(metadata={}, page_content='Mais uma agenda nesse fim de semana com o objetivo de levar o desenvolvimento para nossa Bahia. \nEm reunião com a prefeita marariosmuquem, produtores e em visita à Usina de Muquém do São Francisco, tratamos sobre o projeto da Esc